In [1]:
!pip install python-dotenv
!pip install -q -U google-genai
!pip install python-docx

In [2]:
from dotenv import load_dotenv

from google import genai
from google.genai import types

import os
import re

from docx import Document

In [3]:
load_dotenv()  # Loads from a .env file in your project directory

True

In [4]:
client = genai.Client()

In [5]:
def read_docx(file_path):
    """
    Reads a .docx file and tags bold, italic, and underlined text.

    Args:
        file_path (str): Path to the .docx file

    Returns:
        str: Text with formatting tags applied
    """
    doc = Document(file_path)
    full_text = []

    for paragraph in doc.paragraphs:
        paragraph_text = ""

        for run in paragraph.runs:
            text = run.text

            # Apply tags based on formatting
            # Order matters — apply from inside out
            if run.italic:
                text = f"*{text}*"
            if run.bold:
                text = f"**{text}**"
            if run.underline:
                text = f"__{text}__"

            paragraph_text += text

        full_text.append(paragraph_text)

    return "\n".join(full_text)

In [23]:
def paraphrase_transcript_gemini(file_path, output_path=None,
                                  model="gemini-2.5-flash", temperature=0.6,
                                  max_tokens=8192, max_chars=15000):
    """
    Reads a .docx file and paraphrases it using the Gemini API.

    Args:
        file_path (str): Path to your transcribed .docx file
        model (str): Gemini model to use
        temperature (float): Creativity level (0.4-0.7 recommended)
        max_tokens (int): Max length of response
        max_chars (int): Max characters to read from file

    Returns:
        str: The paraphrased text
    """

  # Check file exists
    if not os.path.exists(file_path):
        return f"Error: File '{file_path}' not found."

    # Check it is a .docx file
    if not file_path.endswith(".docx"):
        return f"Error: File must be a .docx file."

    # Check file is not empty
    if os.path.getsize(file_path) == 0:
        return f"Error: File '{file_path}' is empty."

    try:
        # Read the .docx file
        file_content = read_docx(file_path)

        # Check document has readable text
        if not file_content.strip():
            return "Error: Document appears to have no readable text."

        # Trim if too long
        if len(file_content) > max_chars:
            print(f"Warning: File is long, trimming to {max_chars} characters.")
            file_content = file_content[:max_chars]

        # Build prompts    
        system_instruction ="""
        Act as a research assistant specialized in data anonymization and semantic rewriting.
        Your goal is to completely rewrite the provided transcript so that the original phrasing, unique word choices, 
        and sentence patterns are entirely unidentifiable, preventing the original text from being tracked or matched. 
        Follow these strict rules:
        1. PRESERVE INTENT & FACTUAL DATA: Keep every single core concept, fact, and piece of data intact. Do not lose any analytical meaning.
        2. MAXIMUM WORD ROTATION: Use a completely different vocabulary. 
            Avoid using the same unique verbs, nouns, or adjective combinations found in the original text. Break up and rearrange every sentence.
        3. REMOVE LINGUISTIC FINGERPRINTS: Eliminate personal speech patterns, unique idioms, conversational ticks, 
            and specific sentence structures that could identify the original speaker.
        4. TONE & STYLE: Write in a simple, clear, and conversational style. Avoid heavy academic jargon. 
            Use short, accessible sentences that anyone can easily understand.
        5. NO OBFUSCATION FILLER: Do not add extra fluff just to change the text. Keep it concise.
        6. FIX TRANSCRIPTION ERRORS: Correct any obvious speech-to-text artifacts, filler words (e.g., "um", "uh"), and disjointed grammar.
        7. FORMAT: Maintain the timestamps from the original text. 
            The text uses the following formatting tag: *text* means the original was italic — preserve this on the paraphrased equivalent.
        8. COMPILE COMMON IDEAS: Combine scattered thoughts or overlapping concepts across the conversation into longer, 
            cohesive paragraphs rather than choppy, line-by-line restatements.
        9. REMOVE UNRELATED DIRECTIONS: Eliminate any instructions or off-topic directions that are not directly related to the actual content of the conversation.
        10. OUTPUT ONLY: Return only the newly generated text. Do not include intros, outros, or explanations.

        """
        
        user_prompt = file_content

        # Call Gemini API
        response = client.models.generate_content(
            model=model,
            contents=user_prompt,
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                temperature=temperature,
                max_output_tokens=max_tokens,
            )
        )

        return response.text

    except UnicodeDecodeError:
        return "Error: Could not read file. Make sure it is saved as UTF-8."
    except Exception as e:
        return f"Unexpected error: {str(e)}"


def paraphrase_and_save(file_path,
                         output_path=None, model="gemini-2.5-flash"):
    """
    Saves paraphrased text to a .docx file, converting formatting
    tags back into real bold, italic, and underline formatting.

    Args:
        text (str): Paraphrased text with formatting tags
        output_path (str): Where to save the .docx file
        model (str): Gemini model to use

    Returns:
        str: The paraphrased text
    """

    result = paraphrase_transcript_gemini(
        file_path, model=model
    )

    doc = Document()

    for line in result.split("\n"):
        if not line.strip():
            continue

        paragraph = doc.add_paragraph()

        # Regex to find tagged segments and plain text between them
        # Matches **bold**, *italic*, __underline__, or plain text
        pattern = r'(\*\*.*?\*\*|\*.*?\*|__.*?__|[^*_]+)'
        segments = re.findall(pattern, line)

        for segment in segments:
            run = paragraph.add_run()

            if segment.startswith("**") and segment.endswith("**"):
                run.text = segment[2:-2]
                run.bold = True
            elif segment.startswith("*") and segment.endswith("*"):
                run.text = segment[1:-1]
                run.italic = True
            elif segment.startswith("__") and segment.endswith("__"):
                run.text = segment[2:-2]
                run.underline = True
            else:
                run.text = segment

    doc.save(output_path)
    print(f"Saved with formatting to: {output_path}")

In [25]:
paraphrase_and_save("glenville1.docx", output_path="glenville1_trans.docx")

Saved with formatting to: glenville1_trans.docx
